In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
from db.database import SessionLocal
from datetime import datetime

In [3]:
import pandas as pd
from sqlalchemy.orm import Session
from datetime import datetime

from repository.comment_repository import CommentRepository


class IngestData:
    """
    Responsible ONLY for selecting training data.
    No labeling, no cost, no predictions.
    """

    def __init__(self, db: Session):
        self.repo = CommentRepository(db_session=db)

    def fetch_training_data(self, training_cutoff: datetime) -> pd.DataFrame:
        """
        Time-based training selection (core logic).

        Example:
            - Model v1 → 2024 + early 2025
            - Model v3 → last 90 days only
        """

        # 🔥 fetch only data within time window
        raw_data = self.repo.get_training_view(training_cutoff)

        df = pd.DataFrame(raw_data)

        return df

In [4]:
db = SessionLocal()

training_cutoff = datetime.fromisoformat("2026-01-01")

df = IngestData(db=db).fetch_training_data(training_cutoff=training_cutoff)

In [5]:
df = df.head(10)

In [6]:
df

,comment_text,confidence_score,id,predicted_label,published_at,source
0,عطنا رابطهم لاهنت,None,421,None,2024-07-31 20:13:31+00:00,None
1,لارابط ولا اسم ولا صورة ولاشيء,None,420,None,2024-07-31 23:21:36+00:00,None
2,64B2214DCCGCR,None,419,None,2024-08-01 22:18:19+00:00,None
3,64B2214DCCGCR,None,418,None,2024-08-01 22:20:27+00:00,None
4,كودي 6D5A0497DEGCR شغال 100% برق جربته أنا وا ...,None,417,None,2024-08-01 23:19:05+00:00,None
5,CF00EFD9C2GCR كودي اللي يبي يلعب معي ويكسب,None,416,None,2024-08-02 02:32:16+00:00,None
6,كودي <br>D89FCB27C9GCR<br>اللهم ارزقني وارزق مني,None,415,None,2024-08-03 05:34:56+00:00,None
7,كم المده وتوصل,None,473,None,2024-08-10 01:46:40+00:00,None
8,اخوي شاركت الكود لا اكثر من شخصين ولا جتني محا...,None,472,None,2024-08-10 20:13:28+00:00,None
9,كودي في برق 1394742EA7GCR,None,471,None,2024-08-10 23:47:24+00:00,None


In [7]:
from core.config import get_settings
from store.llm.llm_provider_factory import LLMProviderFactory


settings = get_settings()
llm_factory = LLMProviderFactory(settings)

In [8]:
llm_provider = llm_factory.create(settings.LLM_PROVIDER)
llm_provider.set_generation_model(settings.OPENAI_MODEL_ID)

In [ ]:
import time
from typing import List
from store.llm.providers.openai_provider import OpenAIProvider
from store.llm.templates.comment_intent import SYSTEM_RULES, USER_RULES
import json
import pandas as pd


class DataLabeling:
    def __init__(self, clean_data: pd.DataFrame, llm: OpenAIProvider):
        self.clean_data = clean_data
        self.llm = llm

    def _should_use_model_label(self, row, threshold=0.95):
        label = row.get("predicted_label")
        confidence = row.get("confidence_score")

        if label is None or confidence is None:
            return False

        return float(confidence) >= threshold

    def generate_comment_intent(self):
        predictions = []

        for _, row in self.clean_data.iterrows():

            comment = row["comment_text"]

            # ==================================================
            # 1. MODEL PATH
            # ==================================================
            if self._should_use_model_label(row):

                predictions.append({
                    "comment": row.get("comment"),
                    "intent": row.get("predicted_label"),
                    "source": row.get("source"),
                    "confidence": row.get("confidence_score"),

                    # ⚡ METRICS
                    "latency_ms": row.get("latency_ms"),
                    "cost_usd": row.get("latency_ms"),   
                    "stage": row.get("stage")
                })

                continue

            # ==================================================
            # 2. GPT PATH (FALLBACK)
            # ==================================================
            start = time.time()

            messages = [
                self.llm.construct_prompt(SYSTEM_RULES.safe_substitute(), role="system"),
                self.llm.construct_prompt(USER_RULES.substitute(comment=comment), role="user")
            ]

            response = self.llm.generate_text(chat_history=messages)

            latency_ms = (time.time() - start) * 1000

            raw_text = response["text"]

            try:
                intent = json.loads(
                    raw_text.replace("```json", "").replace("```", "").strip()
                ).get("predicted_intent")
            except json.JSONDecodeError:
                intent = None

            # GPT cost tracking (IMPORTANT)
            prompt_tokens = response.get("prompt_tokens", 0)
            completion_tokens = response.get("completion_tokens", 0)

            cost_usd = self.llm.estimate_cost(prompt_tokens, completion_tokens)

            predictions.append({
                "comment": comment,
                "intent": intent,
                "source": "gpt",
                "confidence": None,
                "raw": raw_text,

                # ⚡ METRICS
                "latency_ms": latency_ms,
                "cost_usd": cost_usd,
                "stage": "training",
            })

        return pd.DataFrame(predictions)


In [10]:
labeled_df = DataLabeling(clean_data=df, llm=llm_provider).generate_comment_intent()

In [11]:
labeled_df

,comment,intent,source,confidence,raw,latency_ms,cost_usd,stage
0,عطنا رابطهم لاهنت,Question,gpt,None,"{\n ""predicted_intent"": ""Question""\n}",2125.073910,0.000025,labeling_gpt
1,لارابط ولا اسم ولا صورة ولاشيء,Complaint,gpt,None,"{\n ""predicted_intent"": ""Complaint""\n}",2209.590912,0.000025,labeling_gpt
2,64B2214DCCGCR,Statement,gpt,None,"```json\n{\n ""predicted_intent"": ""Statement""\...",1359.848976,0.000027,labeling_gpt
3,64B2214DCCGCR,Statement,gpt,None,"```json\n{\n ""predicted_intent"": ""Statement""\...",1124.640942,0.000027,labeling_gpt
4,كودي 6D5A0497DEGCR شغال 100% برق جربته أنا وا ...,Statement,gpt,None,"{\n ""predicted_intent"": ""Statement""\n}",847.829103,0.000029,labeling_gpt
5,CF00EFD9C2GCR كودي اللي يبي يلعب معي ويكسب,Statement,gpt,None,"{\n ""predicted_intent"": ""Statement""\n}",1394.752979,0.000027,labeling_gpt
6,كودي <br>D89FCB27C9GCR<br>اللهم ارزقني وارزق مني,Statement,gpt,None,"{\n ""predicted_intent"": ""Statement""\n}",1027.639151,0.000027,labeling_gpt
7,كم المده وتوصل,Question,gpt,None,"{\n ""predicted_intent"": ""Question""\n}",773.583174,0.000025,labeling_gpt
8,اخوي شاركت الكود لا اكثر من شخصين ولا جتني محا...,Question,gpt,None,"{\n ""predicted_intent"": ""Question""\n}",985.141039,0.000027,labeling_gpt
9,كودي في برق 1394742EA7GCR,Statement,gpt,None,"{\n ""predicted_intent"": ""Statement""\n}",906.996012,0.000026,labeling_gpt
